# Fixed Explainable Hybrid PQD Classification
This notebook includes corrected TCN, proper noise handling, and Grad-CAM.

In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt


## Data (Assuming X_raw, labels already loaded)

In [ ]:

# Dummy placeholder (replace with your dataset loading)
# X_raw shape: (N, 999)
# labels: list of class names

# Example:
# X_raw = ...
# labels = ...

le = LabelEncoder()
y = le.fit_transform(labels)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

def per_sample_standardize(x):
    mu = np.mean(x, axis=1, keepdims=True)
    sig = np.std(x, axis=1, keepdims=True) + 1e-8
    return (x - mu) / sig

X_train = per_sample_standardize(X_train_raw)[..., None]
X_test  = per_sample_standardize(X_test_raw)[..., None]


## Fixed Residual TCN

In [ ]:

def residual_tcn_block(x, filters, dilation_rate, dropout=0.2):
    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation("relu")(h)
    h = layers.Dropout(dropout)(h)

    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(h)
    h = layers.BatchNormalization()(h)

    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding="same")(x)

    h = layers.Add()([x, h])
    return layers.Activation("relu")(h)

def build_model():
    inp = layers.Input((999,1))
    x = inp
    for d in [1,2,4,8,16,32,64]:
        x = residual_tcn_block(x, 64, d)

    x = layers.Conv1D(128, 3, padding="same", activation="relu", name="target_conv_layer")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(len(le.classes_), activation="softmax")(x)
    return Model(inp, out)

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


## Noise Augmentation (Fixed)

In [ ]:

@tf.function
def augment_noise_tf(x, y):
    snr = tf.random.uniform([], 20.0, 50.0)
    xpow = tf.reduce_mean(tf.square(x), axis=[1,2], keepdims=True)
    npow = xpow / (10.0 ** (snr / 10.0))
    noise = tf.random.normal(tf.shape(x), stddev=tf.sqrt(npow))
    return x + noise, y

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(2048).batch(32).map(augment_noise_tf).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)


## Training

In [ ]:

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(patience=3),
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=callbacks)


## Evaluation

In [ ]:

preds = np.argmax(model.predict(X_test), axis=1)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=le.classes_))


## Grad-CAM

In [ ]:

def gradcam_1d(x):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer("target_conv_layer").output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model([x])
        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1))

    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., None]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap)+1e-8)
    return heatmap.numpy()

sample = X_test[:1]
heat = gradcam_1d(sample)

plt.plot(sample[0].squeeze())
plt.imshow(heat[np.newaxis,:], aspect='auto', alpha=0.5)
plt.title("Grad-CAM")
plt.show()
